In [ ]:
import pandas as pd
import re
import os #pour naviguer dans les dossiers
from io import StringIO
import s3fs #pour connecter au bucket

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import umap
from sklearn.datasets import load_digits



In [ ]:


# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

BUCKET_OUT = "aluneau"



In [ ]:
# le fichier pseudonymisé
with fs.open(f"{BUCKET_OUT}/data_eP8/raw/anonymous_answer_2026-05-11.csv", "r") as file_in:
    df0 = pd.read_csv(file_in, sep =",")

df0 = df0.loc[~df0.q45_clé.isin(["Z37Y-6JGC","QNYZ-3MH2"])] # Les identifiants correspondent à des personnes qui ont répondu plusieurs fois, on garde la es réponses les plus anciennes

# le dictionnaire des variables
df_col = pd.read_csv("../le_questionnaire/dico_variable.csv", sep = ",")


# Profil disciplinaire

On s'intéresse au profil disciplinaire des répondants. Cette information est donnée par la question no 110 dont l'intitulé est "1/ Votre/vos champ(s) principaux de recherche ? *Catégories issues de la nomenclature ERC*".
La question étant à choix multiple, l'objectif de l'analyse est de parvenir à une classification des profils de réponse. Pour cela, nous utilisons les algorithmes UMAP et HDBSCAN. Nous procédons enfin à une labellisation manuelle.

Ces différentes étapes sont décrites plus en détails ci-après.

## La répartition des répondants dans la nomenclature de l'ERC

Avant d'aller plus loin, nous allons regarder le nombre de répondants par champ de recherche. 

In [ ]:
def split_multiple_choices(data, column, index, sep = '|'):
    """
    split and explode column with multiple value

    data = a panda dataframe
    column = name of column which we want to split.
    index = column corresponding to id of rows
    sep = by default '|'. Character use to separate values.
    """
    df_split = data.copy()
    df_split[column]= df0.apply(lambda row: row[column].replace(";","|") ,1 )
    df_split[column] = df_split[column].str.split(sep)
    df_explode = df_split.explode(column)
    gb_data = df_explode.groupby([column]).agg(nb = (index, "size")).reset_index()
    gb_data["total"] = data[index].nunique()
    gb_data["freq"] = gb_data.nb/gb_data.total*100
    gb_data["total_freq"] = gb_data.total/gb_data.total*100

    return gb_data, df_explode

In [ ]:
df0["q24_research_fields"]= df0.apply(lambda row: row.q24_research_fields.replace(";","|") ,1 )


In [ ]:
gb_data, df_exp = split_multiple_choices(df0, column='q24_research_fields', index = "q45_clé")
print(gb_data[["q24_research_fields","nb","total","freq"]].sort_values(by="nb", ascending=False).round(1).to_markdown(index=False))

Le tableau ci-dessous montre que les champs de recherche en sciences humaines et sociales sont sans surprise majoritaires. 56~% des répondants situent leur recherche dans le champ des recherches sur le monde social et ses interactions qui, si nous nous référons à la nomenclature de l'ERC, englobe la sociologie, les sciences de l'information et de la communication, les *digital studies*. Vient ensuite le champ intitulé "l'esprit humaine et sa complexité" qui renvoie aux sciences cognitives, à la psychologie ou à la linguistique. Les *études des cultures et des arts* est le troisième champ de recherche le plus représenté avec 37 répondant sur 117 qui se retrouvent dans cette catégoire. Ce champ est plus hétérogène pusiqu'il regroupe aussi bien les approches ethnologiques et anthropologiques que l'histoire de l'art, l'architecture ou la muséologie.

Parmi les champs en SHS les moins représentés, il y les recherches sur les "individus, les marchés et les organisation" (sciences économiques et de la gestion) et "institutions, la gouvernance et les systèmes juridiques" (droits, sciences politiques). Le nombre de personnes (19) qui se situent dans le champ "SH2 Institutions, gouvernance et systèmes juridiques" est presque équivalent aux nombres de chercheurs et chercheuses en "sciences informatiques" (16).

Enfin on a dans notre échantillon quelques représentants des "neurosciences et troubles du système nerveux", de la "physiologie de la santé" et des "sciences du système Terre". Des chercheurs en climatologie ont en effet rejoint dernièrement des laboratoires de Paris 8.


| q24_research_fields                                             |   nb |   total |   freq |
|:----------------------------------------------------------------|-----:|--------:|-------:|
| SH3 Le monde social et ses interactions                         |   66 |     117 |   56.4 |
| SH4 L'esprit humain et sa complexité                            |   39 |     117 |   33.3 |
| SH8 Études des cultures et des arts                             |   37 |     117 |   31.6 |
| SH5 Textes et concepts                                          |   26 |     117 |   22.2 |
| SH7 Mobilité humaine, environnement et espace                   |   24 |     117 |   20.5 |
| SH6 L'étude du passé humain                                     |   24 |     117 |   20.5 |
| SH1 Individus, marchés et organisations                         |   23 |     117 |   19.7 |
| SH2 Institutions, gouvernance et systèmes juridiques            |   19 |     117 |   16.2 |
| PE6 Sciences informatiques et informatique                      |   16 |     117 |   13.7 |
| PE7 Ingénierie des systèmes et de la communication              |    7 |     117 |    6   |
| PE1 Mathématiques                                               |    4 |     117 |    3.4 |
| LS5 Neurosciences et troubles du système nerveux                |    3 |     117 |    2.6 |
| LS4 Physiologie de la santé, de la maladie et du vieillissement |    2 |     117 |    1.7 |
| LS7 Prévention, diagnostic et traitement des maladies humaines  |    2 |     117 |    1.7 |
| PE10 Sciences du Système Terre                                  |    2 |     117 |    1.7 |
| LS8 Biologie environnementale, écologie et évolution            |    1 |     117 |    0.9 |
| LS6 Immunité, infection et immunothérapie                       |    1 |     117 |    0.9 |
| PE9 Sciences de l'Univers                                       |    1 |     117 |    0.9 |
| PE8 Ingénierie des produits et des procédés                     |    1 |     117 |    0.9 |
| LS9 Biotechnologie et ingénierie des biosystèmes                |    1 |     117 |    0.9 |
| PE5 Chimie de synthèse et matériaux                             |    1 |     117 |    0.9 |



In [ ]:
# Initialize the matplotlib figure
fig, ax = plt.subplots(1, figsize=(10, 20))

# Plot the total crashes
sns.set_color_codes("pastel")

for n, col in enumerate(['q24_research_fields']):
    gb_data, df_rf = split_multiple_choices(df0, column=col, index = "q45_clé")
    sns.barplot(x="total_freq", y=col, data=gb_data.sort_values(by="nb", ascending=False),
                label="Non", color="b", ax=ax)
    sns.barplot(x="freq", y=col, data=gb_data,
                label="Oui", color="r", ax=ax)
    titre = df_col.question.loc[df_col.label==col].iloc[0]
    ax.yaxis.set_label_text("")
    ax.xaxis.set_label_text("")
    ax.set_title(titre.replace("<i>","(").replace("</i>",")"))
    

sns.despine(left=True, bottom=True)
#plt.savefig(f"viz/multiple_choice_1.png", bbox_inches='tight', dpi = 200)

## Reduce dimension for research fields

Compte tenu du petit nombre de répondants et du grand nombre de modalités, il nous a semblé pertinent de procéder à un regroupement des champ de recherche proches d'une part et des chercheurs d'autre part. "Proche" s'entend au sens statistique, c'est-à-dire qui sont choisies par les mêmes personnes. Par exemple, selon le tableau ci-dessous, on voit que sur les 19 personnes qui ont sélectionné le champ "Institutions, gouvernance et systèmes juridiques" (SH2) 10 ont également choisi "Individus, marchés et organisations" (SH1) et 15 se situent dans le champ du "monde social et ses interactions" (SH3). En ce sens on peut dire que le champ SH2 est plus proche de SH3 que de SH1.

|                                                      |   SH1 Individus, marchés et organisations |   SH2 Institutions, gouvernance et systèmes juridiques |   SH3 Le monde social et ses interactions |
|:-----------------------------------------------------|------------------------------------------:|-------------------------------------------------------:|------------------------------------------:|
| SH1 Individus, marchés et organisations              |                                        23 |                                                     10 |                                        18 |
| SH2 Institutions, gouvernance et systèmes juridiques |                                        10 |                                                     19 |                                        15 |
| SH3 Le monde social et ses interactions              |                                        18 |                                                     15 |                                        66 |


Nous avons commencé par classer les champs disciplinaires. Pour cela nous utilisons nous appuyons sur sur UMAP et HDBSCAN. Le premier est un algorithme de réduction des dimensions à l'instar d'une analyse en composante principale. Le second est un algorithme de clusterisation.

Il est nécessaire espace géométrique à n dimensions (dans notre cas le nombre de dimensions correspond au nombre de répondants). Pour faciliter l'analyse, nous allons ensuite transformer notre espace vectoriel à 117 dimension en un espace à deux dimensions. On obtient ce qu'on appelle un "plongement vectoriel" (ou *embedding*)[^1].




[^1]: Pour aller plus loin dans la compréhension des embeddings, on pourra se référer à ce tutoriels : [https://developers.google.com/machine-learning/crash-course/embeddings?hl=fr](https://developers.google.com/machine-learning/crash-course/embeddings?hl=fr)

In [ ]:
df_rf = df_exp[["q45_clé", 'q24_research_fields']].copy()
df_rf.loc[df_rf.q24_research_fields.str.contains("SH"), "q24_research_domain"] = "Sciences sociales et humaines"
df_rf.loc[df_rf.q24_research_fields.str.contains("LS"), "q24_research_domain"] = "Sciences de la vie"
df_rf.loc[df_rf.q24_research_fields.str.contains("PE"), "q24_research_domain"] = "Sciences physiques et ingénierie"
df_rf["value"] = 1
df_rf.q24_research_fields.value_counts()

In [ ]:
df_rf1 = df_rf.pivot(index =['q24_research_fields','q24_research_domain'], columns= 'q45_clé', values = "value").fillna(0).reset_index()

In [ ]:
research_fields = df_rf1[df_rf1.columns[2:]].values
m_rf = pd.DataFrame(np.dot(research_fields, research_fields.T), index=[x for x in df_rf1.q24_research_fields], columns=[x for x in df_rf1.q24_research_fields])



#print(m_rf[["SH1 Individus, marchés et organisations","SH2 Institutions, gouvernance et systèmes juridiques","SH3 Le monde social et ses interactions"]].iloc[13:17].to_markdown())

In [ ]:
research_domain_list = [x for x in df_rf1.q24_research_domain]

## Les paramètres de UMAP

Umap est un algorithme dont les résultats dépendent des paramètres définis par l'utilisateur. Ces paramètres sont le nombre de voisins (*n_neighbors*) pris en compte pour calculer la position d'un point au sein de l'espace vectoriel, la distance minimale à respecter entre les points (*min_dist*), le nombre de dimensions à conserver (*n_components*) et la métrique utilisée pour calculer la distance entre les points (*metric*).

La figure ci-dessous montre l'influence du paramètre *n_neighbors* sur la spatialisation des points. Plus la valeur est importante, plus l'algorithme tend à conserver la structure globale. Inversement, plus la valeur est basse, plus l'algorithme préserve les spécificités locales du graphe. Dans notre cas, on voit apparaître plusieurs groupes au sein des sciences sociales et humaines lorsque *n_neighbors* vaut 2, qui disparaissent lorsque ce paramètre vaut 5 ou 10. C'est alors la distinction entre les SHS et les 2 autres "panels disciplinaires" qui ressort. le graphique devient même difficle à "lire" lorsqu'on passe a une valeur de 10.

![](viz/umap_researchfield.jpeg)

Dans notre cas, il nous semble donc plus intéressant de définir la valeur de *n_neighbors* à 2 afin de saisir les différences entre les disciplines SHS, sachant que la distinction en les SHS d'un côté et les sciences de la vie et les sciences physiques nous est déjà donnée par le biais de la nomenclature ERC. 

## Clusterisation avec HDBSCAN

Une fois la réduction des dimensions opérées, nous allons utiliser l'algorithme HDBSCAN pour regrouper une "clusterisation" (regroupement) des discipines. Là encore, HDBSCAN es tun algorithme paramétrique dont le résulstat, c'est-à-dire le nombre de cluster, dépend de la taille minimale des clusters (*min_cluster_size*) et la densité de représentants d'un cluster (*min_sample*) pour déterminer si un point en fait partie ou non. Le paramètre *min_cluster_size* va avoir un effet sur le nombre de cluster identifier : plus la taille minimale est petite, plus le nombre de cluster risque d'être grand. L'autre paramètre (*min_sample*) joue sur les contraintes que doit respecter un point pour faire partie d'un cluster. Ces contraintes correspond au nombre de points appartenant à un cluster qui l'entourent pour qu'il soit lui même considérer comme membre du cluster. Par exemple, si *min_sample* vaut 1, alors il suffit qu'un point soit proche d'un seul membre du cluster A pour qu'il soit intégrer à ce dernier. Par contre si *min_sample* vaut 10, il faut alors qu'il soit au centre d'un groupe de 10 points du cluster A pour être ajouté à ce cluster.






In [ ]:
import umap
from sklearn.datasets import load_digits


fig, ax = plt.subplots(1, 3, figsize=(20, 8))
nns = [2, 5, 10]
i = -1
for n_neighbors in nns:
    i+=1
    fit = umap.UMAP(n_neighbors=n_neighbors, min_dist=0.1, n_components=2, random_state=42, metric = 'cosine')
    u = fit.fit_transform(research_fields);
    sns.scatterplot(x=u[:,0], y=u[:,1],  ax=ax[i], hue= [x for x in df_rf1.q24_research_domain])
    ax[i].set_title(f'n_neighbors={n_neighbors}')

#fig.savefig("viz/umap_researchfield.jpeg", format="jpeg", dpi = 300)

In [ ]:
embedding = umap.UMAP(n_neighbors=2, min_dist=0.1, n_components=2, random_state=42, metric = 'cosine').fit_transform(research_fields)


In [ ]:

import hdbscan
import sklearn.cluster as cluster
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score


In [ ]:
labels = hdbscan.HDBSCAN(
    min_samples=1,
    min_cluster_size=2, gen_min_span_tree=True
).fit(embedding)


In [ ]:
dict_clusters = {}
for n, x in enumerate(df_rf1.q24_research_fields):
    dict_clusters[x] = labels.labels_[n]


for x in range(-1,len(set(labels.labels_))):
    print("Cluster :", x)
    for v in dict_clusters:
        if dict_clusters[v] == x:
            print(v)

In [ ]:
label_cluster = {0 : "Humanités", #histoire philosophie
                 1: "Sciences cognitives, neurosciences", 
                 2: "Institutions, normes et économies",
                3: "Espaces et sociétés", #sociologie, géo, SIC
                4: "Maths info",
                5: "Biologie, maladie",
                6: "Ecologie et biosystèmes"}

In [ ]:
list_cluster = []
for x in range(len(set(labels.labels_))):
    #print("Cluster :", x)
    for v in dict_clusters:
        if dict_clusters[v] == x:
            columns={"q24_research_fields": v,
                         "hdbscan_cluster": x,
                         "id_cluster": x,
                         "lab_cluster": label_cluster[x]
                        }
            list_cluster.append(columns)

df_cluster = pd.DataFrame.from_dict(list_cluster).dropna()
         

In [ ]:
df_cluster.to_csv("../data/clusterisation/research_field_cluster.csv", sep =",", index = False)

In [ ]:
c_palette = ["red", "blue", "green", "yellow", "black", "pink", "purple","brown"]


In [ ]:
fig, ax = plt.subplots(1, figsize=(14, 14))
sns.scatterplot(x=embedding[:,0], y=embedding[:,1],  ax=ax, hue= [label_cluster[x] for x in labels.labels_], palette=c_palette )

fig.savefig("viz/umap_hdbscancluster.jpeg", format="jpeg", dpi = 300)

Après plusieurs essais, nous avons retenus les paramètres suivants :

* *min_sample* = 1
* *min_cluster_size* = 2 (sachant qu'un cluster, par définition, ne peut pas avoir une taille inférieure à 2)

Ces paramètres conduisent à l'identification de sept clusters. Il n'y a aucun outliers. Parmi les sept clusters, on retrouve les groupes de disciplines de SHS observables sur la représentation graphique des résultats obtenus avec UMAP. Ils correspondent aux groupes des "humanités", "droit, de l'économie et des sciences politiques", et des "sciences sociales". Les quatre autres clusters rassemblent les points qui, sur la projection, se trouvent au centre. On a les disciplines des "sciences cognitives et neurosciences", des "mathématiques et sciences informatiques", de la "biologie" et de l'"écologie".

* Cluster 0 : "Humanités"
  - SH4 L'esprit humain et sa complexité
  - SH5 Textes et concepts
  - SH6 L'étude du passé humain
  - SH8 Études des cultures et des arts
* Cluster 2 : "Institutions, normes et économie",
  - SH1 Individus, marchés et organisations
  - SH2 Institutions, gouvernance et systèmes juridiques
* Cluster 3 : "Espaces et sociétés"
  - SH3 Le monde social et ses interactions
  - SH7 Mobilité humaine, environnement et espace


* Cluster 1 : "Psychologie et neurosciences"
  - LS4 Physiologie de la santé, de la maladie et du vieillissement
  - LS5 Neurosciences et troubles du système nerveux
  - PE5 Chimie de synthèse et matériaux
* Cluster 4 : "Maths info"
  - PE1 Mathématiques
  - PE6 Sciences informatiques et informatique
  - PE7 Ingénierie des systèmes et de la communication
* Cluster 5 : "Biologie et santé"
  - LS6 Immunité, infection et immunothérapie
  - LS7 Prévention, diagnostic et traitement des maladies humaines
  - PE10 Sciences du Système Terre
* CLuster 6 : "Écologie et biosystèmes"
  - LS8 Biologie environnementale, écologie et évolution
  - LS9 Biotechnologie et ingénierie des biosystèmes
  - PE8 Ingénierie des produits et des procédés
  - PE9 Sciences de l'Univers

![](viz/umap_hdbscancluster.jpeg)

```mermaid
graph TD;
SHS --> 0;
SHS --> 2;
SHS --> 3;
0 --> SH4;
0 --> SH5;
0 --> SH6;
0 --> SH8;
2 --> SH1;
2 --> SH2;
3 --> SH3;
3 --> SH7;
Non-SHS --> 1;
Non-SHS --> 4;
Non-SHS --> 5;
Non-SHS --> 6;
1 --> LS4;
1 --> LS5;
1 --> PE5;
4 --> PE1;
4 --> PE6;
4 --> PE7;
5 --> LS6;
5 --> LS7;
5 --> PE10;
6 --> LS8;
6 --> LS9;
6 --> PE8;
6 --> PE9;
```

In [ ]:
df_rf

In [ ]:

# Initialize the matplotlib figure
fig, ax = plt.subplots(1, figsize=(10, 8))

# Plot the total crashes
sns.set_color_codes("pastel")


df_cluster1 = df_rf.merge(df_cluster, on = ["q24_research_fields"], how ="left")

for n, col in enumerate(['lab_cluster']):
    gb_data = df_cluster1.drop_duplicates(["q45_clé","lab_cluster"]).groupby(["lab_cluster"]).agg(nb = ("q45_clé", "size"))
    gb_data["total_nb"] = df_rf.q45_clé.nunique()
    sns.barplot(x="total_nb", y=col, data=gb_data.sort_values(by="nb", ascending=False),
                label="Non", color="b", ax=ax)
    sns.barplot(x="nb", y=col, data=gb_data,
                label="Oui", color="r", ax=ax)
    titre = "Tailles des groupes disciplinaires"
    ax.yaxis.set_label_text("")
    ax.xaxis.set_label_text("")
    ax.set_title(titre.replace("<i>","(").replace("</i>",")"))


sns.despine(left=True, bottom=True)


fig.savefig("viz/cluster_field_frequency.jpeg", format="jpeg", dpi = 300)

# Clusterisation des individus

Après le regroupement des champs disciplinaires, nous allons utilisés les mêmes algorithmes pour, cette fois, identifier les individus aux profils disciplinaires proches. L'exploration des données a mis en évidence l'existence de chercheurs qui "cochent" toutes les disciplines ou qui ont répondu deux fois (sans sélectionner le même nombre de disciplines). Nous commençons par enlevés ces lignes qui créent du bruit.

Ensuite, nous avons commencé par une approche "non-supervisée", puis au fur et à mesure de la classification des individus, nous avons utilisés les labels pour contruire des modèles semi-supervisé. Au final, nous avons identifiés neuf aires disciplinaires. Ces informations sont contenues dans les fichiers "tableau_cluster_ind.txt" et "tableau_cluster.txt".

Le premier groupe que nous avons isolé est celui des "géographes" que nous avons nommé "Environnement et espace". Les répondants appartenant à ce groupe ont pour point commun d'avoir tous choisi deux champ disciplinaires -- "Le monde social et ses interactions" (SH3) et "Mobilité humaine, environnement et espace" (SH7) -- comme le montre leurs profilfs de réponse ci-dessous. Un autre groupe distinctif est celui qui rassemblent les répondants qui s'identifient aux sciences informatiques (PE6). Le groupe "cognition et raisonnement" se caractrisent par des répondants qui se situent dans les champs des études sur "Le monde social et ses interactions" (SH3) et "l'esprit humain et sa complexité" (SH4). Ils se distinguent des chercheurs classés dans le groupe de la "psychologies" et qui ont également sélectionné le champ SH4 (l'esprit humain et sa complexité), mais ne se retrouvent pas dans les disciplines du "monde social et ses interactions". Le groupe "Recherche et création" rassemble les chercheurs qui ont répondu uniquement "études des cultures et des arts"(SH8) et dans le groupe "Histoire" ceux qui n'ont coché.

Les groupes des "humanités numériques" et des "sciences humùmaines et sociales" sont les deux groupes les plus importants, mais aussi les plus hétérogènes dans la mesure où ils ne se caractérisent pas par un profil unique comme dans le cas des chercheurs classés dans les catégories "Environnement et espace" ou "Sciences informatiques". Si les chercheurs classés en "humanités numériques" se rapprochent de ceux en "cognition et raisonnement" ou en "psychologie" du fait qu'ils s'intéressent à "l'esprit humain et sa complexité", ils s'en distinguent par le fait qu'ils se rattachent à plusieurs autres champs disciplinaires des SHS comme "le monde social et ses interactions" (SH3), "les textes et concepts" (SH5) ou l'"étude des cultures et des arts" (SH8). Quant aux répondants faisant partie de la catégorie "Sciences hmaines et sociales", ils se situe à la frontière de plusieurs domaines des SHS. C'est dans ce groupe que l'on retrouve les chercheurs qui étudient les "individus, les organisations et les marché" (SH1, économie) ainsi que les "institutions, la gouvernance et les systèmes juridiques" (SH2) que ce soit du point de vue du droit, de la sociologie, de l'histoire ou en comparant des contextes culturels différents. On observe à l'intérieur du groupe SHS la présence d'historiens qui se démarque des chercheurs classés dans le groupe "Histoire" du fait qu'ils se situent aussi dans les études du mondes social et ses interactions, là où les seconds se reconnaissent uniquement dans l'"étude du passé humain".


| Research areas                |   count |
|:------------------------------|--------:|
| Sciences humaines et sociales |      35 |
| Humanités (numériques)        |      22 |
| Psychologie                   |      14 |
| Information et communication  |      10 |
| Recherche et création         |       9 |
| Environnement et espace       |       8 |
| Cognition et raisonnement     |       6 |
| Sciences informatiques        |       6 |
| Histoire                      |       5 |




In [ ]:
#### df_rf5.to_csv("embedding_indiv_research_fields.csv", sep = ",", index = False)

df_rfi = pd.read_csv("../data/clusterisation/research_field_ind_VF.csv", sep=",")


#print(df_rfi.area.value_counts().to_markdown())


In [ ]:
#%%capture cap
dict_area = dict(zip(df_rfi.q45_clé, df_rfi.area))
list_area = [x for x in df_rfi.area.unique()]

with open("tableau_cluster.txt", 'w') as fout: 
    for x in list_area:
        fout.write(f"### {x}\n\n")
        name_column = df_rfi.columns[1:22]
        abrev = "|".join([re.search(r"\w*\d+", x).group() for x in name_column ])
        table_frame = "|".join(["---:" for col in name_column])
        fout.write(f"|{abrev}|\n")
        fout.write(f"|{table_frame}|\n")

        compteur = 0
        for v in dict_area:
            if dict_area[v] == x:
                compteur+=1
                dtmp0 = df_rfi.merge(affil, on = "q45_clé", how = "left")
                dtmp = dtmp0[dtmp0.columns[35]].loc[dtmp0.q45_clé == v].values
                dtmp1 = dtmp0[dtmp0.columns[1:22]].loc[dtmp0.q45_clé == v].values
                fout.write(f'|{"|".join([str(x).replace("1.0","**1.0**") for x in dtmp1[0]])}|\n')
        fout.write("\n\n")

### Environnement et espace

|SH1|SH2|SH3|SH5|SH6|SH7|SH8|PE7|PE8|PE9|PE10|LS4|LS6|LS7|LS8|LS9|SH4|LS5|PE6|PE1|PE5|
|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
|0.0|0.0|**1.0**|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|0.0|0.0|**1.0**|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|0.0|0.0|**1.0**|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|0.0|0.0|**1.0**|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|0.0|0.0|**1.0**|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|0.0|0.0|**1.0**|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|0.0|0.0|**1.0**|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|


### Information et communication

|SH1|SH2|SH3|SH5|SH6|SH7|SH8|PE7|PE8|PE9|PE10|LS4|LS6|LS7|LS8|LS9|SH4|LS5|PE6|PE1|PE5|
|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|
|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|


### Cognition et raisonnement

|SH1|SH2|SH3|SH5|SH6|SH7|SH8|PE7|PE8|PE9|PE10|LS4|LS6|LS7|LS8|LS9|SH4|LS5|PE6|PE1|PE5|
|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|
|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|
|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|
|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|
|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|
|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|


### Sciences informatiques

|SH1|SH2|SH3|SH5|SH6|SH7|SH8|PE7|PE8|PE9|PE10|LS4|LS6|LS7|LS8|LS9|SH4|LS5|PE6|PE1|PE5|
|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|
|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|
|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|**1.0**|0.0|
|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|
|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|
|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|


### Recherche et création

|SH1|SH2|SH3|SH5|SH6|SH7|SH8|PE7|PE8|PE9|PE10|LS4|LS6|LS7|LS8|LS9|SH4|LS5|PE6|PE1|PE5|
|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|
|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|
|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|


### Humanités (numériques)

|SH1|SH2|SH3|SH5|SH6|SH7|SH8|PE7|PE8|PE9|PE10|LS4|LS6|LS7|LS8|LS9|SH4|LS5|PE6|PE1|PE5|
|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
|0.0|0.0|**1.0**|0.0|0.0|0.0|**1.0**|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|**1.0**|0.0|0.0|
|0.0|**1.0**|**1.0**|0.0|**1.0**|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|
|0.0|0.0|**1.0**|0.0|**1.0**|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|**1.0**|0.0|
|0.0|**1.0**|**1.0**|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|
|0.0|0.0|**1.0**|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|**1.0**|**1.0**|0.0|
|0.0|0.0|**1.0**|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|
|0.0|**1.0**|**1.0**|0.0|**1.0**|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|0.0|0.0|**1.0**|**1.0**|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|
|0.0|0.0|**1.0**|**1.0**|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|
|0.0|0.0|**1.0**|**1.0**|**1.0**|**1.0**|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|**1.0**|0.0|0.0|
|0.0|0.0|0.0|**1.0**|**1.0**|**1.0**|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|
|**1.0**|**1.0**|**1.0**|**1.0**|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|**1.0**|0.0|0.0|
|0.0|0.0|**1.0**|**1.0**|**1.0**|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|
|0.0|0.0|**1.0**|**1.0**|**1.0**|**1.0**|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|
|0.0|0.0|0.0|**1.0**|**1.0**|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|0.0|0.0|**1.0**|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|
|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|
|0.0|0.0|0.0|**1.0**|**1.0**|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|
|**1.0**|0.0|**1.0**|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|
|0.0|**1.0**|0.0|**1.0**|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|**1.0**|**1.0**|**1.0**|0.0|
|0.0|0.0|**1.0**|**1.0**|**1.0**|0.0|**1.0**|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|
|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|


### Sciences humaines et sociales

|SH1|SH2|SH3|SH5|SH6|SH7|SH8|PE7|PE8|PE9|PE10|LS4|LS6|LS7|LS8|LS9|SH4|LS5|PE6|PE1|PE5|
|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
|0.0|0.0|**1.0**|0.0|**1.0**|**1.0**|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|0.0|0.0|**1.0**|0.0|**1.0**|**1.0**|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|0.0|0.0|**1.0**|0.0|**1.0**|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|0.0|0.0|**1.0**|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|**1.0**|0.0|**1.0**|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|**1.0**|0.0|**1.0**|**1.0**|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|**1.0**|**1.0**|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|0.0|**1.0**|**1.0**|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|**1.0**|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|**1.0**|0.0|**1.0**|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|**1.0**|0.0|**1.0**|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|**1.0**|**1.0**|**1.0**|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|**1.0**|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|**1.0**|**1.0**|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|**1.0**|0.0|**1.0**|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|0.0|**1.0**|**1.0**|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|0.0|**1.0**|**1.0**|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|**1.0**|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|**1.0**|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|**1.0**|0.0|**1.0**|**1.0**|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|**1.0**|**1.0**|**1.0**|0.0|**1.0**|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|**1.0**|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|**1.0**|**1.0**|**1.0**|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|**1.0**|**1.0**|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|0.0|**1.0**|**1.0**|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|**1.0**|**1.0**|**1.0**|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|0.0|0.0|**1.0**|**1.0**|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|0.0|0.0|0.0|**1.0**|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|0.0|0.0|**1.0**|**1.0**|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|0.0|0.0|**1.0**|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|0.0|0.0|0.0|**1.0**|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|0.0|0.0|0.0|**1.0**|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|


### Histoire

|SH1|SH2|SH3|SH5|SH6|SH7|SH8|PE7|PE8|PE9|PE10|LS4|LS6|LS7|LS8|LS9|SH4|LS5|PE6|PE1|PE5|
|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
|0.0|0.0|0.0|0.0|**1.0**|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|
|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|
|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|


### Psychologie

|SH1|SH2|SH3|SH5|SH6|SH7|SH8|PE7|PE8|PE9|PE10|LS4|LS6|LS7|LS8|LS9|SH4|LS5|PE6|PE1|PE5|
|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|
|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|
|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|
|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|
|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|
|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|
|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|
|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|
|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|**1.0**|0.0|0.0|
|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|
|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|
|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|
|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|
|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|**1.0**|**1.0**|0.0|0.0|0.0|


